In [43]:
import requests
import pandas as pd
from functools import reduce

API_KEY = "kMWwLaP6bX0Qwf2zdKR4cE4hVt4r2cDQjpRZJeCg"

def fetch_series(series_id):
    url = f"https://api.eia.gov/v2/seriesid/{series_id}?api_key={API_KEY}"
    response = requests.get(url)
    response.raise_for_status()

    rows = response.json()["response"]["data"]

    return {
        row["period"]: row["value"]
        for row in rows
    }

In [44]:
state_series = {
    "Texas": "PET.MCRFPTX2.M",
    "New Mexico": "PET.MCRFPNM2.M",
    "North Dakota": "PET.MCRFPND2.M",
    "Colorado": "PET.MCRFPCO2.M",
    "Alaska": "PET.MCRFPAK2.M",
    "California": "PET.MCRFPCA2.M"
}

In [45]:
all_states = []

for state, series_id in state_series.items():

    print("Fetching:", state)

    data = fetch_series(series_id)

    temp_df = pd.DataFrame(
        data.items(),
        columns=["month", "production_thousand_bpd"]
    )

    temp_df["state"] = state

    all_states.append(temp_df)

Fetching: Texas
Fetching: New Mexico
Fetching: North Dakota
Fetching: Colorado
Fetching: Alaska
Fetching: California


In [46]:
state_production = pd.concat(
    all_states,
    ignore_index=True
)

In [47]:
state_production["month"] = pd.to_datetime(
    state_production["month"]
).dt.to_period("M")

In [48]:
state_production = state_production.sort_values(
    ["month", "state"]
).reset_index(drop=True)

In [49]:
state_production = state_production[
    state_production["month"] >= "2006-09"
]

In [50]:
state_production['state'].value_counts()

state
Alaska          238
California      238
Colorado        238
New Mexico      238
North Dakota    238
Texas           238
Name: count, dtype: int64

In [51]:
state_production = state_production.pivot(
    index="month",
    columns="state",
    values="production_thousand_bpd"
).reset_index()

print(state_production.shape)
state_production.head()

(238, 7)


state,month,Alaska,California,Colorado,New Mexico,North Dakota,Texas
0,2006-09,655,610,68,164,112,1071
1,2006-10,714,612,70,163,113,1079
2,2006-11,655,610,69,169,114,1084
3,2006-12,785,607,64,165,114,1078
4,2007-01,772,605,61,162,114,1058


In [53]:
state_production["total_production"] = state_production[
    ["Alaska", "California", "Colorado", "New Mexico", "North Dakota", "Texas"]
].sum(axis=1)

In [54]:
state_production.to_csv("state_production.csv", index = False)